# Building Your Own MCP Server

The previous notebooks in this series consumed MCP servers as black boxes — the SRE agent in Notebook 03 used `sre_mcp_server.py` without explaining how it worked inside. This notebook fills that gap.

**What you'll build:** A weather MCP server with two tools, then connect it to a Claude agent using the same patterns from earlier notebooks.

**What you'll learn:**
- The JSON-RPC protocol that MCP runs on
- How to define tools with JSON Schema
- What schema constraints matter for client compatibility
- How Claude discovers and calls your tools

**Prerequisites:** Python 3.11+, basic familiarity with the Claude Agent SDK (see Notebook 00).

## What is MCP?

The **Model Context Protocol** is an open standard for connecting AI models to external tools and data sources. Instead of hard-coding tool logic into your agent, you expose it through a standardized server that any MCP-compatible client can use.

### The protocol in one paragraph

An MCP server runs as a separate process. The client (Claude Agent SDK) spawns it, sends JSON-RPC messages over stdin, and reads responses from stdout. The client first calls `initialize` to handshake, then `tools/list` to discover available tools, then `tools/call` whenever it wants to invoke one. Your server handles these three message types — that's the whole protocol.

### Transport options

- **stdio** — the server is a local subprocess, connected via stdin/stdout. Simple, no network required. This is what we use here.
- **HTTP+SSE** — the server is a remote service. Needed when multiple clients share a server, or when the server runs in the cloud.

### The three primitives

MCP servers can expose three things:
- **Tools** — callable functions (what we focus on here)
- **Resources** — readable data sources (files, database records)
- **Prompts** — reusable prompt templates

Tools are what you need to give Claude the ability to *do* things.

In [ ]:
%pip install -U claude-agent-sdk python-dotenv

## MCP Server Anatomy

An MCP server must respond to three JSON-RPC methods:

```
Client                          Server
  │                               │
  │──── initialize ──────────────►│  Handshake: agree on protocol version
  │◄─── {protocolVersion, ...} ───│
  │                               │
  │──── tools/list ──────────────►│  Discovery: what tools exist?
  │◄─── [{name, description, ...}]│
  │                               │
  │──── tools/call ──────────────►│  Execution: run this tool with these args
  │◄─── {content: [{text: ...}]} ─│
```

Each tool is described by a JSON Schema object:

```python
{
    "name": "get_weather",
    "description": "Get current weather for a city.",
    "inputSchema": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "City name, e.g. Tokyo or New York"
            }
        },
        "required": ["city"]
    }
}
```

### Critical schema constraint

**Use primitive types only** (`string`, `number`, `integer`, `boolean`). Parameters with `array` or `object` types are silently dropped by some MCP clients — your tool will appear to work but those parameters will be missing. If you need a list, accept a pipe-separated string and split it in your handler:

```python
# Bad: "type": "array"  — silently dropped by some clients
# Good: "type": "string", "description": "Pipe-separated city names, e.g. Tokyo|Paris"
cities = cities_str.split("|")
```

## Building a Weather MCP Server

We'll implement two tools:
- `get_weather(city)` — returns mock weather data for a city
- `convert_temperature(value, from_unit, to_unit)` — converts between Celsius, Fahrenheit, and Kelvin

The server is a plain Python script. We use `%%writefile` to write it to disk, then the Agent SDK spawns it as a subprocess.

In [ ]:
%%writefile weather_mcp_server.py
#!/usr/bin/env python3
"""
Weather MCP Server — stdio transport.

Implements the MCP JSON-RPC protocol over stdin/stdout.
Handles: initialize, tools/list, tools/call

Usage:
    python weather_mcp_server.py
"""

import asyncio
import json
import sys
from typing import Any

# ---------------------------------------------------------------------------
# Tool definitions (JSON Schema)
# ---------------------------------------------------------------------------

TOOLS = [
    {
        "name": "get_weather",
        "description": (
            "Get the current weather conditions for a city. "
            "Returns temperature in Celsius, humidity, wind speed, and a short description."
        ),
        "inputSchema": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "City name, e.g. Tokyo, London, or New York",
                }
            },
            "required": ["city"],
        },
    },
    {
        "name": "convert_temperature",
        "description": (
            "Convert a temperature value between Celsius (C), Fahrenheit (F), and Kelvin (K). "
            "Use this when the user asks for a temperature in a different unit."
        ),
        "inputSchema": {
            "type": "object",
            "properties": {
                "value": {
                    "type": "number",
                    "description": "The numeric temperature value to convert",
                },
                "from_unit": {
                    "type": "string",
                    "description": "Source unit: C, F, or K",
                },
                "to_unit": {
                    "type": "string",
                    "description": "Target unit: C, F, or K",
                },
            },
            "required": ["value", "from_unit", "to_unit"],
        },
    },
]

# ---------------------------------------------------------------------------
# Tool handlers
# ---------------------------------------------------------------------------

# Mock weather data — in a real server, replace with an API call
WEATHER_DATA = {
    "tokyo": {"temp_c": 18, "humidity": 65, "wind_kph": 12, "description": "Partly cloudy"},
    "london": {"temp_c": 10, "humidity": 80, "wind_kph": 20, "description": "Overcast"},
    "new york": {"temp_c": 14, "humidity": 55, "wind_kph": 18, "description": "Clear"},
    "sydney": {"temp_c": 24, "humidity": 60, "wind_kph": 15, "description": "Sunny"},
    "paris": {"temp_c": 12, "humidity": 72, "wind_kph": 10, "description": "Light rain"},
}


def handle_get_weather(city: str) -> dict[str, Any]:
    key = city.lower().strip()
    data = WEATHER_DATA.get(key)

    if data is None:
        # Return an error the agent can see and reason about
        return {
            "content": [
                {
                    "type": "text",
                    "text": (
                        f"No weather data found for '{city}'. "
                        f"Available cities: {', '.join(WEATHER_DATA.keys())}."
                    ),
                }
            ],
            "isError": True,
        }

    result = (
        f"Weather in {city.title()}:\n"
        f"  Temperature: {data['temp_c']}°C\n"
        f"  Humidity: {data['humidity']}%\n"
        f"  Wind: {data['wind_kph']} kph\n"
        f"  Conditions: {data['description']}"
    )
    return {"content": [{"type": "text", "text": result}]}


def handle_convert_temperature(value: float, from_unit: str, to_unit: str) -> dict[str, Any]:
    from_unit = from_unit.upper().strip()
    to_unit = to_unit.upper().strip()
    valid_units = {"C", "F", "K"}

    if from_unit not in valid_units or to_unit not in valid_units:
        return {
            "content": [
                {
                    "type": "text",
                    "text": f"Invalid unit. Use C, F, or K. Got from_unit={from_unit!r}, to_unit={to_unit!r}.",
                }
            ],
            "isError": True,
        }

    if from_unit == to_unit:
        return {"content": [{"type": "text", "text": f"{value}°{from_unit}"}]}

    # Convert to Celsius first, then to target
    if from_unit == "F":
        celsius = (value - 32) * 5 / 9
    elif from_unit == "K":
        celsius = value - 273.15
    else:
        celsius = value

    if to_unit == "F":
        result = celsius * 9 / 5 + 32
    elif to_unit == "K":
        result = celsius + 273.15
    else:
        result = celsius

    text = f"{value}°{from_unit} = {result:.2f}°{to_unit}"
    return {"content": [{"type": "text", "text": text}]}


# ---------------------------------------------------------------------------
# JSON-RPC dispatch
# ---------------------------------------------------------------------------


def send_response(response: dict[str, Any]) -> None:
    """Write a JSON-RPC response to stdout."""
    sys.stdout.write(json.dumps(response) + "\n")
    sys.stdout.flush()


def send_error(req_id: Any, code: int, message: str) -> None:
    send_response({"jsonrpc": "2.0", "id": req_id, "error": {"code": code, "message": message}})


def handle_request(request: dict[str, Any]) -> None:
    method = request.get("method", "")
    req_id = request.get("id")
    params = request.get("params", {})

    if method == "initialize":
        send_response({
            "jsonrpc": "2.0",
            "id": req_id,
            "result": {
                "protocolVersion": "2024-11-05",
                "capabilities": {"tools": {}},
                "serverInfo": {"name": "weather-tools", "version": "1.0.0"},
            },
        })

    elif method == "notifications/initialized":
        # Notification — no response needed
        pass

    elif method == "tools/list":
        send_response({"jsonrpc": "2.0", "id": req_id, "result": {"tools": TOOLS}})

    elif method == "tools/call":
        name = params.get("name", "")
        args = params.get("arguments", {})

        try:
            if name == "get_weather":
                result = handle_get_weather(city=args.get("city", ""))
            elif name == "convert_temperature":
                result = handle_convert_temperature(
                    value=args.get("value", 0),
                    from_unit=args.get("from_unit", "C"),
                    to_unit=args.get("to_unit", "C"),
                )
            else:
                result = {
                    "content": [{"type": "text", "text": f"Unknown tool: {name}"}],
                    "isError": True,
                }
        except Exception as exc:
            result = {
                "content": [{"type": "text", "text": f"Tool error: {exc}"}],
                "isError": True,
            }

        send_response({"jsonrpc": "2.0", "id": req_id, "result": result})

    else:
        send_error(req_id, -32601, f"Method not found: {method}")


async def main() -> None:
    """Read JSON-RPC requests from stdin, dispatch, respond to stdout."""
    loop = asyncio.get_running_loop()
    reader = asyncio.StreamReader()
    protocol = asyncio.StreamReaderProtocol(reader)
    await loop.connect_read_pipe(lambda: protocol, sys.stdin)

    while True:
        line = await reader.readline()
        if not line:
            break
        line = line.decode("utf-8").strip()
        if not line:
            continue
        try:
            request = json.loads(line)
            handle_request(request)
        except json.JSONDecodeError as exc:
            send_error(None, -32700, f"Parse error: {exc}")
        except Exception as exc:
            print(f"Unhandled error: {exc}", file=sys.stderr)


if __name__ == "__main__":
    asyncio.run(main())


## Testing the Server Standalone

Before wiring the server into an agent, verify it speaks the protocol correctly. We'll start it as a subprocess, send raw JSON-RPC messages over its stdin, and read the responses.

This is the same exchange that happens inside the Claude Agent SDK — you're just doing it manually.

In [ ]:
import json
import subprocess
import sys


def send_message(proc: subprocess.Popen, message: dict) -> dict:
    """Send one JSON-RPC message and return the parsed response."""
    line = json.dumps(message) + "\n"
    proc.stdin.write(line)
    proc.stdin.flush()
    response_line = proc.stdout.readline()
    return json.loads(response_line)


proc = subprocess.Popen(
    [sys.executable, "weather_mcp_server.py"],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    text=True,
)

try:
    # Step 1: Handshake
    init_response = send_message(proc, {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "initialize",
        "params": {"protocolVersion": "2024-11-05", "clientInfo": {"name": "test"}},
    })
    print("initialize response:")
    print(json.dumps(init_response, indent=2))

    # Step 2: Discover tools
    list_response = send_message(proc, {
        "jsonrpc": "2.0",
        "id": 2,
        "method": "tools/list",
        "params": {},
    })
    tools = list_response["result"]["tools"]
    print(f"\ntools/list response: {len(tools)} tools found")
    for t in tools:
        print(f"  - {t['name']}: {t['description'][:60]}...")

    # Step 3: Call a tool
    call_response = send_message(proc, {
        "jsonrpc": "2.0",
        "id": 3,
        "method": "tools/call",
        "params": {"name": "get_weather", "arguments": {"city": "Tokyo"}},
    })
    print("\ntools/call get_weather(Tokyo):")
    print(call_response["result"]["content"][0]["text"])

    # Step 4: Call the conversion tool
    conv_response = send_message(proc, {
        "jsonrpc": "2.0",
        "id": 4,
        "method": "tools/call",
        "params": {
            "name": "convert_temperature",
            "arguments": {"value": 25, "from_unit": "C", "to_unit": "F"},
        },
    })
    print("\ntools/call convert_temperature(25C -> F):")
    print(conv_response["result"]["content"][0]["text"])

finally:
    proc.terminate()
    proc.wait()

Expected output:
```
initialize response:
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "protocolVersion": "2024-11-05",
    "capabilities": {"tools": {}},
    "serverInfo": {"name": "weather-tools", "version": "1.0.0"}
  }
}

tools/list response: 2 tools found
  - get_weather: Get the current weather conditions for a city. ...
  - convert_temperature: Convert a temperature value between Celsius ...

tools/call get_weather(Tokyo):
Weather in Tokyo:
  Temperature: 18°C
  Humidity: 65%
  Wind: 12 kph
  Conditions: Partly cloudy

tools/call convert_temperature(25C -> F):
25°C = 77.00°F
```

## Connecting to the Claude Agent SDK

Now we wire the server into the Agent SDK. The `mcp_servers` key in `ClaudeAgentOptions` tells the SDK which subprocess to spawn. The SDK handles the `initialize` / `tools/list` handshake automatically — Claude sees your tools alongside its native capabilities.

The `allowed_tools` list uses the format `mcp__{server_name}__{tool_name}`, where `server_name` is the key you gave in `mcp_servers`.

In [ ]:
import os
import sys
from pathlib import Path

from claude_agent_sdk import (
    AssistantMessage,
    ClaudeAgentOptions,
    ResultMessage,
    TextBlock,
    ToolUseBlock,
    query,
)

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
if not ANTHROPIC_API_KEY:
    raise ValueError("ANTHROPIC_API_KEY environment variable not set")

MODEL = "claude-haiku-4-5"
MCP_SERVER_PATH = Path("weather_mcp_server.py").resolve()
assert MCP_SERVER_PATH.exists(), f"Server not found at {MCP_SERVER_PATH}"

options = ClaudeAgentOptions(
    model=MODEL,
    system_prompt=(
        "You are a helpful weather assistant. "
        "Use the available tools to answer weather questions. "
        "Always convert temperatures to the unit the user asks for."
    ),
    mcp_servers={
        "weather": {
            "command": sys.executable,
            "args": [str(MCP_SERVER_PATH)],
        }
    },
    allowed_tools=[
        "mcp__weather__get_weather",
        "mcp__weather__convert_temperature",
    ],
)

prompt = "What's the weather in Tokyo? Convert the temperature to Fahrenheit for me."
print(f"Prompt: {prompt}\n")

async for message in query(prompt=prompt, options=options):
    if isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, TextBlock) and block.text.strip():
                print(block.text.strip())
            elif isinstance(block, ToolUseBlock):
                tool_name = block.name.replace("mcp__weather__", "")
                print(f"[tool] {tool_name}({block.input})")
    elif isinstance(message, ResultMessage) and message.is_error:
        print(f"ERROR: {message.result}")

Expected output:
```
Prompt: What's the weather in Tokyo? Convert the temperature to Fahrenheit for me.

[tool] get_weather({'city': 'Tokyo'})
[tool] convert_temperature({'value': 18, 'from_unit': 'C', 'to_unit': 'F'})
The current weather in Tokyo is partly cloudy with a temperature of 18°C (64.40°F),
65% humidity, and winds at 12 kph.
```

Claude discovered both tools from `tools/list`, chose which ones to call, constructed the right arguments, and chained them without being told to. The tool descriptions drove the decision — `convert_temperature`'s description mentions Fahrenheit, so Claude knew to call it when the user asked for an F reading.

## Schema Design Best Practices

Tool descriptions are how Claude decides *when* and *how* to call your tools. The schema is part of the interface — treat it like a function signature that Claude reads.

### Do

- **Use primitive types for all parameters** (`string`, `number`, `integer`, `boolean`). This is the most important rule — array and object types are silently dropped by some clients.
- **Write descriptions that explain when to use the tool**, not just what it does. "Use this when the user asks for X" beats "Does X".
- **Name parameters unambiguously.** `from_unit` and `to_unit` are better than `unit1` and `unit2`.
- **List concrete examples in descriptions** — `"e.g. Tokyo, London, or New York"` reduces hallucinated inputs.
- **Use `required` honestly.** Only mark parameters required if the tool cannot run without them.

### Don't

- **Don't use `array` or `object` parameter types.** Use pipe-separated strings instead:
  ```python
  # In schema
  "cities": {"type": "string", "description": "Pipe-separated cities: Tokyo|London|Paris"}
  # In handler
  cities = [c.strip() for c in cities_str.split("|")]
  ```
- **Don't make tools too broad.** A tool named `do_database_thing` that conditionally reads or writes is hard for Claude to reason about. Separate read and write tools get better call accuracy.
- **Don't swallow errors silently.** Return `isError: True` with a message — Claude can recover from errors it can see.

### Enums via description

When a parameter has a small fixed set of valid values, list them explicitly in the description:
```python
"from_unit": {
    "type": "string",
    "description": "Temperature unit: C (Celsius), F (Fahrenheit), or K (Kelvin)"
}
```
Some schemas support `"enum": ["C", "F", "K"]` — but pipe-separating in the description works universally.

## Error Handling in MCP Tools

When a tool fails, return `isError: True` in the result. Claude sees this as a tool failure and can adjust — retry with different parameters, try a different tool, or explain the limitation to the user.

The next cell demonstrates what happens when Claude calls `get_weather` for a city not in our dataset.

In [ ]:
prompt = "What's the weather in Buenos Aires?"
print(f"Prompt: {prompt}\n")

async for message in query(prompt=prompt, options=options):
    if isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, TextBlock) and block.text.strip():
                print(block.text.strip())
            elif isinstance(block, ToolUseBlock):
                tool_name = block.name.replace("mcp__weather__", "")
                print(f"[tool] {tool_name}({block.input})")
    elif isinstance(message, ResultMessage) and message.is_error:
        print(f"ERROR: {message.result}")

Expected output:
```
Prompt: What's the weather in Buenos Aires?

[tool] get_weather({'city': 'Buenos Aires'})
I'm sorry, but I don't have weather data for Buenos Aires. The cities I currently
have data for are: tokyo, london, new york, sydney, and paris. Would you like
the weather for any of these cities instead?
```

The server returned `isError: True` with a message listing available cities. Claude read the error, surfaced it naturally to the user, and offered alternatives — without crashing or silently returning wrong data.

**The key pattern:** errors returned via `isError: True` are visible to Claude and recoverable. Unhandled exceptions that crash the subprocess are not — Claude sees a connection failure with no useful message.

## Production Server Template

The weather server above is minimal by design — easy to read, easy to explain. A production server needs a few more things:

- **Logging to stderr** (stdout is reserved for JSON-RPC)
- **Input validation** before calling external APIs
- **Graceful error handling** around every tool call
- **A clear tool registration pattern** that scales to 10+ tools

The template below shows these patterns applied to a generic three-tool server.

In [ ]:
%%writefile production_mcp_template.py
#!/usr/bin/env python3
"""
Production MCP Server Template — stdio transport.

Patterns demonstrated:
- Logging to stderr (never stdout — that's for JSON-RPC)
- Input validation before calling external services
- Structured error returns (isError: True) vs crashes
- Tool registry dict for clean dispatch at scale
"""

import asyncio
import json
import logging
import sys
from typing import Any

# Log to stderr — stdout is the JSON-RPC channel
logging.basicConfig(
    stream=sys.stderr,
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
)
logger = logging.getLogger("mcp-server")


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------


def ok(text: str) -> dict[str, Any]:
    """Return a successful tool result."""
    return {"content": [{"type": "text", "text": text}]}


def err(text: str) -> dict[str, Any]:
    """Return a tool error visible to Claude."""
    return {"content": [{"type": "text", "text": text}], "isError": True}


# ---------------------------------------------------------------------------
# Tool handlers — one function per tool
# ---------------------------------------------------------------------------


def handle_search(query: str, max_results: str = "5") -> dict[str, Any]:
    """Example read tool: search a data source."""
    if not query.strip():
        return err("query cannot be empty")

    try:
        limit = int(max_results)
    except ValueError:
        return err(f"max_results must be an integer, got: {max_results!r}")

    if limit < 1 or limit > 50:
        return err("max_results must be between 1 and 50")

    # Replace with real search logic
    logger.info("search query=%r limit=%d", query, limit)
    results = [f"Result {i+1} for '{query}'" for i in range(min(limit, 3))]
    return ok("\n".join(results))


def handle_create_item(name: str, category: str, description: str = "") -> dict[str, Any]:
    """Example write tool: create a record."""
    if not name.strip():
        return err("name cannot be empty")

    valid_categories = {"task", "note", "event"}
    if category not in valid_categories:
        return err(f"category must be one of: {', '.join(sorted(valid_categories))}")

    # Replace with real persistence logic
    logger.info("create_item name=%r category=%r", name, category)
    return ok(f"Created {category} '{name}' (id: mock-123)")


def handle_summarize(text: str, format: str = "bullets") -> dict[str, Any]:
    """Example transformation tool: process text."""
    if not text.strip():
        return err("text cannot be empty")

    valid_formats = {"bullets", "paragraph", "tldr"}
    if format not in valid_formats:
        return err(f"format must be one of: {', '.join(sorted(valid_formats))}")

    # Replace with real summarization logic
    logger.info("summarize format=%r chars=%d", format, len(text))
    return ok(f"[{format}] Summary of {len(text)} chars: (mock output)")


# ---------------------------------------------------------------------------
# Tool registry — maps name -> (handler, schema)
# Adding a new tool = add one entry here + one handler function above
# ---------------------------------------------------------------------------

TOOL_REGISTRY: dict[str, tuple[Any, dict]] = {
    "search": (
        handle_search,
        {
            "name": "search",
            "description": "Search the knowledge base. Returns matching records.",
            "inputSchema": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search terms"},
                    "max_results": {
                        "type": "string",
                        "description": "Max results to return (1-50, default 5)",
                    },
                },
                "required": ["query"],
            },
        },
    ),
    "create_item": (
        handle_create_item,
        {
            "name": "create_item",
            "description": "Create a new item in the system. Use for tasks, notes, and events.",
            "inputSchema": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "Item name"},
                    "category": {
                        "type": "string",
                        "description": "Item type: task, note, or event",
                    },
                    "description": {
                        "type": "string",
                        "description": "Optional longer description",
                    },
                },
                "required": ["name", "category"],
            },
        },
    ),
    "summarize": (
        handle_summarize,
        {
            "name": "summarize",
            "description": "Summarize a block of text in a given format.",
            "inputSchema": {
                "type": "object",
                "properties": {
                    "text": {"type": "string", "description": "Text to summarize"},
                    "format": {
                        "type": "string",
                        "description": "Output format: bullets, paragraph, or tldr",
                    },
                },
                "required": ["text"],
            },
        },
    ),
}

TOOLS = [schema for _, schema in TOOL_REGISTRY.values()]


# ---------------------------------------------------------------------------
# JSON-RPC layer — generic, no changes needed when adding tools
# ---------------------------------------------------------------------------


def send_response(response: dict[str, Any]) -> None:
    sys.stdout.write(json.dumps(response) + "\n")
    sys.stdout.flush()


def send_error(req_id: Any, code: int, message: str) -> None:
    send_response({"jsonrpc": "2.0", "id": req_id, "error": {"code": code, "message": message}})


def handle_request(request: dict[str, Any]) -> None:
    method = request.get("method", "")
    req_id = request.get("id")
    params = request.get("params", {})

    if method == "initialize":
        send_response({
            "jsonrpc": "2.0",
            "id": req_id,
            "result": {
                "protocolVersion": "2024-11-05",
                "capabilities": {"tools": {}},
                "serverInfo": {"name": "production-template", "version": "1.0.0"},
            },
        })
    elif method == "notifications/initialized":
        pass
    elif method == "tools/list":
        send_response({"jsonrpc": "2.0", "id": req_id, "result": {"tools": TOOLS}})
    elif method == "tools/call":
        name = params.get("name", "")
        args = params.get("arguments", {})

        if name not in TOOL_REGISTRY:
            result = err(f"Unknown tool: {name}. Available: {', '.join(TOOL_REGISTRY)}")
        else:
            handler, _ = TOOL_REGISTRY[name]
            try:
                result = handler(**args)
            except TypeError as exc:
                # Missing or unexpected keyword argument
                result = err(f"Invalid arguments for {name}: {exc}")
            except Exception as exc:
                logger.exception("Tool %r raised", name)
                result = err(f"Tool error: {exc}")

        send_response({"jsonrpc": "2.0", "id": req_id, "result": result})
    else:
        send_error(req_id, -32601, f"Method not found: {method}")


async def main() -> None:
    loop = asyncio.get_running_loop()
    reader = asyncio.StreamReader()
    protocol = asyncio.StreamReaderProtocol(reader)
    await loop.connect_read_pipe(lambda: protocol, sys.stdin)

    logger.info("MCP server started, %d tools registered", len(TOOL_REGISTRY))
    while True:
        line = await reader.readline()
        if not line:
            break
        line = line.decode("utf-8").strip()
        if not line:
            continue
        try:
            request = json.loads(line)
            handle_request(request)
        except json.JSONDecodeError as exc:
            send_error(None, -32700, f"Parse error: {exc}")
        except Exception as exc:
            logger.exception("Unhandled error handling request")


if __name__ == "__main__":
    asyncio.run(main())


The key design choices in the production template:

1. **`ok()` and `err()` helpers** — return consistent shapes without repeating the nested dict structure on every line.

2. **Tool registry dict** — maps tool name to `(handler, schema)`. Adding a new tool is two steps: write the handler, add one entry to `TOOL_REGISTRY`. The JSON-RPC dispatch loop never changes.

3. **`handler(**args)` dispatch** — passes the MCP arguments dict directly as kwargs. Catches `TypeError` for missing/unexpected params separately from `Exception` for runtime failures — two different error messages for two different problems.

4. **All logging to `sys.stderr`** — stdout is exclusively the JSON-RPC channel. Any print or log statement that goes to stdout will corrupt the protocol.

## Next Steps

This notebook covered the stdio transport and the Tools primitive. From here:

**HTTP+SSE transport** — Use `mcp.server.fastmcp` or a plain ASGI app to expose your server over HTTP. Required when multiple clients share one server, or when the server runs remotely.

**Resources** — Expose readable data (files, database rows, API responses) via `resources/list` and `resources/read`. Claude can pull resources into context without you building explicit fetch tools.

**Prompts** — Expose reusable prompt templates via `prompts/list` and `prompts/get`. Useful for standardizing how Claude approaches recurring tasks.

**Publishing** — Once your server is stable, you can list it in the MCP registry so others can use it. The spec and registry are at [modelcontextprotocol.io](https://modelcontextprotocol.io).

The pattern scales: start with two or three tools, validate with a real agent, then expand. The `TOOL_REGISTRY` structure in the production template above handles 50 tools as cleanly as 3.